# Speech Emotion Recognition: training notebook

Trains a CNN that recognises 8 emotions (angry, calm, disgust, fear, happy, neutral, sad, surprise) from speech, using four public datasets (RAVDESS, CREMA-D, TESS, SAVEE).

**How to run:** in Colab choose *Runtime > Change runtime type > T4 GPU*, then *Runtime > Run all*. It takes about 30 to 60 minutes. At the end it downloads a zip file with the trained model and the results.

**What makes the result trustworthy:** the data is split **by speaker**. A person whose voice is used for testing never appears in the training data. Without this, a model can look much better than it really is.

## 1. Setup

In [ ]:
!pip -q install kagglehub librosa soundfile

In [ ]:
import json, os, random, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

EMOTIONS = ["angry", "calm", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
EMOTION_TO_ID = {e: i for i, e in enumerate(EMOTIONS)}
OUT = "/content/artifacts"
os.makedirs(OUT, exist_ok=True)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 2. Shared feature code
The same file is used by the server, so training and prediction always compute the same 162 numbers per clip.

In [ ]:
%%writefile features.py
"""Audio features shared by the training notebook and the server.

Both must use exactly this code, otherwise the model sees different
numbers at prediction time than it saw during training.
"""
import librosa
import numpy as np

SAMPLE_RATE = 22050
DURATION = 2.5   # seconds of audio used per clip
OFFSET = 0.6     # skip the first 0.6 s (often silence)
N_FEATURES = 162  # 1 zero crossing + 12 chroma + 20 MFCC + 1 RMS + 128 mel


def load_audio(path):
    """Read an audio file as a mono signal at the common sample rate."""
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, duration=DURATION, offset=OFFSET)
    return audio


def extract_features(audio, sr=SAMPLE_RATE):
    """Turn an audio signal into one vector of 162 numbers."""
    zcr = np.mean(librosa.feature.zero_crossing_rate(y=audio).T, axis=0)
    chroma = np.mean(librosa.feature.chroma_stft(y=audio, sr=sr).T, axis=0)
    mfcc = np.mean(librosa.feature.mfcc(y=audio, sr=sr).T, axis=0)
    rms = np.mean(librosa.feature.rms(y=audio).T, axis=0)
    mel = np.mean(librosa.feature.melspectrogram(y=audio, sr=sr).T, axis=0)
    return np.hstack([zcr, chroma, mfcc, rms, mel]).astype(np.float32)

In [ ]:
from features import SAMPLE_RATE, N_FEATURES, load_audio, extract_features

## 3. Download the four datasets
These are public datasets on Kaggle. `kagglehub` downloads them without needing an account.

In [ ]:
import kagglehub

paths = {
    "ravdess": kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio"),
    "crema":   kagglehub.dataset_download("ejlok1/cremad"),
    "tess":    kagglehub.dataset_download("ejlok1/toronto-emotional-speech-set-tess"),
    "savee":   kagglehub.dataset_download("ejlok1/surrey-audiovisual-expressed-emotion-savee"),
}
for name, p in paths.items():
    n = sum(f.lower().endswith(".wav") for _, _, fs in os.walk(p) for f in fs)
    print(f"{name:8s} {n:5d} wav files")

## 4. Read the labels from the file names
Each dataset stores the emotion and the speaker in the file name. We turn them into one table with the columns `path`, `emotion`, `speaker`.

In [ ]:
RAVDESS_CODES = {"01": "neutral", "02": "calm", "03": "happy", "04": "sad",
                 "05": "angry", "06": "fear", "07": "disgust", "08": "surprise"}
CREMA_CODES = {"ANG": "angry", "DIS": "disgust", "FEA": "fear",
               "HAP": "happy", "NEU": "neutral", "SAD": "sad"}
TESS_CODES = {"angry": "angry", "disgust": "disgust", "fear": "fear", "happy": "happy",
              "neutral": "neutral", "sad": "sad", "ps": "surprise"}
SAVEE_CODES = {"a": "angry", "d": "disgust", "f": "fear", "h": "happy",
               "n": "neutral", "sa": "sad", "su": "surprise"}


def parse_file(dataset, path):
    """Return (emotion, speaker) for one file, or None if the name is not recognised."""
    stem = os.path.splitext(os.path.basename(path))[0]
    if dataset == "ravdess":            # 03-01-05-01-02-01-12.wav (emotion = 3rd, actor = last)
        parts = stem.split("-")
        if len(parts) != 7 or parts[2] not in RAVDESS_CODES:
            return None
        return RAVDESS_CODES[parts[2]], "ravdess_" + parts[6]
    if dataset == "crema":              # 1001_DFA_ANG_XX.wav
        parts = stem.split("_")
        if len(parts) < 3 or parts[2] not in CREMA_CODES:
            return None
        return CREMA_CODES[parts[2]], "crema_" + parts[0]
    if dataset == "tess":               # OAF_back_angry.wav
        parts = stem.split("_")
        label = parts[-1].lower()
        if label not in TESS_CODES:
            return None
        return TESS_CODES[label], "tess_" + parts[0].upper()
    if dataset == "savee":              # DC_sa01.wav
        speaker, rest = stem.split("_", 1)
        label = rest.rstrip("0123456789")
        if label not in SAVEE_CODES:
            return None
        return SAVEE_CODES[label], "savee_" + speaker.upper()


rows = []
for dataset, root in paths.items():
    for folder, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith(".wav"):
                path = os.path.join(folder, f)
                parsed = parse_file(dataset, path)
                if parsed:
                    rows.append({"path": path, "dataset": dataset,
                                 "emotion": parsed[0], "speaker": parsed[1]})

df = pd.DataFrame(rows)
# some Kaggle downloads contain the same folder twice, so keep one copy of each file name
df["file"] = df["path"].map(os.path.basename)
df = df.drop_duplicates(subset=["dataset", "file"]).reset_index(drop=True)
df["label"] = df["emotion"].map(EMOTION_TO_ID)
print(len(df), "files,", df["speaker"].nunique(), "speakers")
print(df.groupby(["dataset", "emotion"]).size().unstack(fill_value=0))

## 5. Split by speaker: train 70%, validation 15%, test 15%
The test speakers are never seen during training. The loop picks a split in which every emotion appears in all three parts.

In [ ]:
def speaker_split(df, seed):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    train_idx, rest_idx = next(gss1.split(df, groups=df["speaker"]))
    rest = df.iloc[rest_idx]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    val_rel, test_rel = next(gss2.split(rest, groups=rest["speaker"]))
    return df.iloc[train_idx], rest.iloc[val_rel], rest.iloc[test_rel]


for seed in range(SEED, SEED + 200):
    train_df, val_df, test_df = speaker_split(df, seed)
    if all(part["emotion"].nunique() == len(EMOTIONS) for part in (train_df, val_df, test_df)):
        break
else:
    raise RuntimeError("no split found with all emotions in every part")

print("split seed:", seed)
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} {len(part):5d} files, {part['speaker'].nunique():3d} speakers")
assert not set(train_df.speaker) & set(test_df.speaker)
assert not set(train_df.speaker) & set(val_df.speaker)
assert not set(val_df.speaker) & set(test_df.speaker)
print("no speaker appears in more than one part")

## 6. Extract features
Training clips are also augmented with added noise and a pitch shift, which gives the model 3 versions of each clip. Validation and test clips are used as they are.

In [ ]:
def add_noise(audio):
    return audio + 0.005 * np.random.uniform() * np.max(np.abs(audio)) * np.random.randn(len(audio))


def pitch_up(audio):
    import librosa
    return librosa.effects.pitch_shift(audio, sr=SAMPLE_RATE, n_steps=2)


def features_plain(path):
    return [extract_features(load_audio(path))]


def features_augmented(path):
    audio = load_audio(path)
    return [extract_features(a) for a in (audio, add_noise(audio), pitch_up(audio))]


def build(part, fn):
    results = Parallel(n_jobs=-1)(delayed(fn)(p) for p in part["path"])
    X, y = [], []
    for feats, label in zip(results, part["label"]):
        for f in feats:
            X.append(f); y.append(label)
    return np.array(X), np.array(y)


X_train, y_train = build(train_df, features_augmented)
X_val, y_val = build(val_df, features_plain)
X_test, y_test = build(test_df, features_plain)
print(X_train.shape, X_val.shape, X_test.shape)
assert X_train.shape[1] == N_FEATURES

Scale each feature to a similar range. The scaling numbers come from the training data only.

In [ ]:
mean = X_train.mean(axis=0)
scale = X_train.std(axis=0) + 1e-8
norm = lambda X: (X - mean) / scale
Xtr, Xva, Xte = norm(X_train), norm(X_val), norm(X_test)

## 7. Baseline: a Random Forest
A simple model on the same features. The CNN should beat it, otherwise it was not worth the extra work.

In [ ]:
rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED, class_weight="balanced")
rf.fit(Xtr, y_train)
rf_pred = rf.predict(Xte)
rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred, average="weighted")
print(f"Random Forest on unseen speakers: accuracy {rf_acc:.3f}, weighted F1 {rf_f1:.3f}")

## 8. The CNN

In [ ]:
def build_model():
    L = tf.keras.layers
    return tf.keras.Sequential([
        L.Input(shape=(N_FEATURES, 1)),
        L.Conv1D(256, 5, padding="same", activation="relu"),
        L.MaxPooling1D(5, strides=2, padding="same"),
        L.Conv1D(256, 5, padding="same", activation="relu"),
        L.MaxPooling1D(5, strides=2, padding="same"),
        L.Conv1D(128, 5, padding="same", activation="relu"),
        L.MaxPooling1D(5, strides=2, padding="same"),
        L.Dropout(0.3),
        L.Conv1D(64, 5, padding="same", activation="relu"),
        L.MaxPooling1D(5, strides=2, padding="same"),
        L.Flatten(),
        L.Dense(64, activation="relu"),
        L.Dropout(0.4),
        L.Dense(len(EMOTIONS), activation="softmax"),
    ])


model = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

weights = compute_class_weight("balanced", classes=np.arange(len(EMOTIONS)), y=y_train)
class_weight = dict(enumerate(weights))
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5),
]
history = model.fit(Xtr[..., None], y_train, validation_data=(Xva[..., None], y_val),
                    epochs=100, batch_size=128, class_weight=class_weight,
                    callbacks=callbacks, verbose=2)

## 9. Results on the unseen test speakers
These numbers are the honest result of the project.

In [ ]:
probs = model.predict(Xte[..., None], verbose=0)
pred = probs.argmax(axis=1)
cnn_acc = accuracy_score(y_test, pred)
cnn_f1 = f1_score(y_test, pred, average="weighted")
print(f"CNN            on unseen speakers: accuracy {cnn_acc:.3f}, weighted F1 {cnn_f1:.3f}")
print(f"Random Forest  on unseen speakers: accuracy {rf_acc:.3f}, weighted F1 {rf_f1:.3f}")
print()
print(classification_report(y_test, pred, target_names=EMOTIONS, digits=3))

cm = confusion_matrix(y_test, pred, labels=range(len(EMOTIONS)))
cm_pct = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(EMOTIONS)), EMOTIONS, rotation=45, ha="right")
ax.set_yticks(range(len(EMOTIONS)), EMOTIONS)
for i in range(len(EMOTIONS)):
    for j in range(len(EMOTIONS)):
        ax.text(j, i, f"{cm_pct[i, j]:.2f}", ha="center", va="center",
                color="white" if cm_pct[i, j] > 0.5 else "black", fontsize=8)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Test speakers (share of each actual emotion)")
fig.colorbar(im); fig.tight_layout()
fig.savefig(f"{OUT}/confusion_matrix.png", dpi=150)
plt.show()

## 10. Save everything the server needs

In [ ]:
model.save(f"{OUT}/emotion_cnn.keras")
np.savez(f"{OUT}/scaler.npz", mean=mean, scale=scale)
json.dump(EMOTIONS, open(f"{OUT}/labels.json", "w"))
metrics = {
    "split_seed": int(seed),
    "train_files": int(len(train_df)), "val_files": int(len(val_df)), "test_files": int(len(test_df)),
    "train_speakers": int(train_df.speaker.nunique()),
    "val_speakers": int(val_df.speaker.nunique()),
    "test_speakers": int(test_df.speaker.nunique()),
    "cnn": {"accuracy": float(cnn_acc), "weighted_f1": float(cnn_f1)},
    "random_forest": {"accuracy": float(rf_acc), "weighted_f1": float(rf_f1)},
    "per_class": classification_report(y_test, pred, target_names=EMOTIONS, output_dict=True),
}
json.dump(metrics, open(f"{OUT}/metrics.json", "w"), indent=2)

!cd /content && zip -qr emotion_artifacts.zip artifacts && ls -la /content/emotion_artifacts.zip
from google.colab import files
files.download("/content/emotion_artifacts.zip")